# 📈 Adım 8 — Performans Grafikleri

Bu notebook'ta 6 zorunlu grafik üretilmektedir:

| # | Grafik | Neyi gösteriyor? |
|---|--------|------------------|
| 1 | ROC Curve | Sensitivity vs Specificity dengesi |
| 2 | Precision-Recall Curve | Dengeli olmayan sınıflar için daha bilgilendirici |
| 3 | Confusion Matrix | Hangi tip hatayı kaç kez yapıyoruz? |
| 4 | Feature Importance | Hangi özellikler kararı etkiliyor? |
| 5 | Calibration Curve | Model olasılıkları gerçekçi mi? |
| 6 | Model Karşılaştırma | RF / ET / GB / Ensemble yan yana |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, VotingClassifier)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, f1_score, accuracy_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from mrmr import mrmr_classif
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Kütüphaneler hazır ✓')


In [ ]:
normal = pd.read_csv('../data/normal_radiomics.csv')
papil  = pd.read_csv('../data/papilodem_radiomics.csv')
normal['label'] = 0; papil['label'] = 1
papil['PatientIndex'] = papil['PatientIndex'] + 1000  # Normal ve papilödem ID'leri örtüşüyor, offset ekle
df = pd.concat([normal, papil], ignore_index=True)
feature_cols = [c for c in df.columns if c.startswith('Feature_')]

def hasta_bazinda_bol(df, test_ratio=0.20, val_ratio=0.10, random_state=42):
    rng = np.random.RandomState(random_state)
    hasta_etiket = df.groupby('PatientIndex')['label'].first()
    train_idx, val_idx, test_idx = [], [], []
    for sinif in [0, 1]:
        hastalar = hasta_etiket[hasta_etiket == sinif].index.tolist()
        rng.shuffle(hastalar)
        n = len(hastalar)
        n_test = max(1, int(n * test_ratio))
        n_val  = max(1, int(n * val_ratio))
        test_idx  += hastalar[:n_test]
        val_idx   += hastalar[n_test:n_test + n_val]
        train_idx += hastalar[n_test + n_val:]
    return (df[df['PatientIndex'].isin(train_idx)].copy(),
            df[df['PatientIndex'].isin(val_idx)].copy(),
            df[df['PatientIndex'].isin(test_idx)].copy())

class RadyomikOnIsleme:
    def __init__(self, vt=0.01, ct=0.95):
        self.imputer = SimpleImputer(strategy='median')
        self.var_sel = VarianceThreshold(threshold=vt)
        self.scaler  = RobustScaler()
        self.feats_var = self.feats_corr = None
    def fit(self, X, fn):
        X = np.where(np.isinf(X), np.nan, X)
        X = self.imputer.fit_transform(X)
        self.var_sel.fit(X)
        m = self.var_sel.get_support()
        self.feats_var = [f for f, v in zip(fn, m) if v]
        X = X[:, m]
        up = pd.DataFrame(X, columns=self.feats_var).corr().abs().where(
            np.triu(np.ones((len(self.feats_var),)*2), k=1).astype(bool))
        drop = [c for c in up.columns if any(up[c] > 0.95)]
        self.feats_corr = [f for f in self.feats_var if f not in drop]
        ki = [self.feats_var.index(f) for f in self.feats_corr]
        self.scaler.fit(X[:, ki]); return self
    def transform(self, X, fn):
        X = np.where(np.isinf(X), np.nan, X)
        X = self.imputer.transform(X)
        X = pd.DataFrame(X, columns=fn)[self.feats_var].values
        X = pd.DataFrame(X, columns=self.feats_var)[self.feats_corr].values
        return self.scaler.transform(X)
    def fit_transform(self, X, fn): self.fit(X, fn); return self.transform(X, fn)

def optuna_en_iyi_params(X_tr, y_tr, groups, model_adi, n_trials=50):
    inner_cv = StratifiedGroupKFold(n_splits=3)
    def objective(trial):
        if model_adi in ['RF','ET']:
            Cls = RandomForestClassifier if model_adi=='RF' else ExtraTreesClassifier
            model = Cls(
                n_estimators=trial.suggest_int('n_estimators',50,500),
                max_depth=trial.suggest_int('max_depth',3,20),
                min_samples_split=trial.suggest_int('min_samples_split',2,20),
                min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10),
                max_features=trial.suggest_categorical('max_features',['sqrt','log2']),
                random_state=42, n_jobs=-1)
        else:
            model = GradientBoostingClassifier(
                n_estimators=trial.suggest_int('n_estimators',50,300),
                max_depth=trial.suggest_int('max_depth',2,8),
                learning_rate=trial.suggest_float('learning_rate',0.01,0.3,log=True),
                subsample=trial.suggest_float('subsample',0.5,1.0),
                min_samples_split=trial.suggest_int('min_samples_split',2,20),
                random_state=42)
        scores = [f1_score(y_tr[vi], model.fit(X_tr[ti], y_tr[ti]).predict(X_tr[vi]), average='macro')
                  for ti, vi in inner_cv.split(X_tr, y_tr, groups)]
        return np.mean(scores)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

MRMR_K = 10
print('Setup hazır ✓')


In [ ]:
# Grafik için tek representatif split kullanıyoruz (split_no=0, random_state=42)
# NOT: Metrik hesaplamalarında 20 split ortalaması alınır.
# Grafikler için en 'ortalama' performanslı split seçmek en doğrusudur.

print('En temsili split aranıyor (en ortalama F1)...')

f1_kayit = []
for split_no in range(20):
    train_df, val_df, test_df = hasta_bazinda_bol(df, random_state=split_no*7+42)
    pipe = RadyomikOnIsleme()
    X_tr = pipe.fit_transform(train_df[feature_cols].values, feature_cols)
    X_te = pipe.transform(test_df[feature_cols].values, feature_cols)
    y_tr = train_df['label'].values
    y_te = test_df['label'].values
    groups = train_df['PatientIndex'].values
    proc_cols = pipe.feats_corr
    feats = mrmr_classif(X=pd.DataFrame(X_tr, columns=proc_cols), y=pd.Series(y_tr), K=MRMR_K)
    X_tr_m = pd.DataFrame(X_tr, columns=proc_cols)[feats].values
    X_te_m = pd.DataFrame(X_te, columns=proc_cols)[feats].values
    params = {m: optuna_en_iyi_params(X_tr_m, y_tr, groups, m) for m in ['RF','ET','GB']}
    rf = RandomForestClassifier(**params['RF'], random_state=42, n_jobs=-1)
    et = ExtraTreesClassifier(**params['ET'], random_state=42, n_jobs=-1)
    gb = GradientBoostingClassifier(**params['GB'], random_state=42)
    ensemble = VotingClassifier(estimators=[('RF',rf),('ET',et),('GB',gb)], voting='soft')
    ensemble.fit(X_tr_m, y_tr)
    cal = CalibratedClassifierCV(ensemble, method='sigmoid', cv=3)
    cal.fit(X_tr_m, y_tr)
    prob = cal.predict_proba(X_te_m)[:, 1]
    f1_val = f1_score(y_te, (prob >= 0.5).astype(int), average='macro')
    f1_kayit.append({'split': split_no, 'f1': f1_val,
                     'train_df': train_df, 'test_df': test_df,
                     'X_tr_m': X_tr_m, 'X_te_m': X_te_m,
                     'y_tr': y_tr, 'y_te': y_te,
                     'feats': feats, 'proc_cols': proc_cols,
                     'rf': rf, 'et': et, 'gb': gb,
                     'ensemble': ensemble, 'cal': cal, 'prob_cal': prob})
    print(f'Split {split_no+1:2d}: F1={f1_val:.4f}')

# En ortalama: medyana en yakın F1
all_f1 = [r['f1'] for r in f1_kayit]
median_f1 = np.median(all_f1)
best_split = min(f1_kayit, key=lambda r: abs(r['f1'] - median_f1))
print(f'\n✓ Seçilen split: {best_split["split"]+1} (F1={best_split["f1"]:.4f}, medyan={median_f1:.4f})')

# Grafiklerde kullanılacak değişkenler
S = best_split
X_tr_m = S['X_tr_m']; X_te_m = S['X_te_m']
y_tr = S['y_tr']; y_te = S['y_te']
rf = S['rf']; et = S['et']; gb = S['gb']
cal_ens = S['cal']
feats = S['feats']

y_prob_ens = S['prob_cal']
y_prob_rf  = rf.predict_proba(X_te_m)[:, 1]
y_prob_et  = et.predict_proba(X_te_m)[:, 1]
y_prob_gb  = gb.predict_proba(X_te_m)[:, 1]
print('Tüm olasılıklar hesaplandı ✓')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- GRAFİK 1: ROC Curve ---
modeller = [('RF', y_prob_rf, 'steelblue'),
             ('ET', y_prob_et, 'salmon'),
             ('GB', y_prob_gb, 'seagreen'),
             ('Ensemble', y_prob_ens, 'darkorange')]

for isim, prob, renk in modeller:
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc = roc_auc_score(y_te, prob)
    ls = '-' if isim == 'Ensemble' else '--'
    lw = 3 if isim == 'Ensemble' else 1.5
    axes[0].plot(fpr, tpr, color=renk, linestyle=ls, linewidth=lw,
                 label=f'{isim} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, linewidth=1)
axes[0].fill_between(*roc_curve(y_te, y_prob_ens)[:2],
                      alpha=0.08, color='darkorange')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
axes[0].set_title('Grafik 1 — ROC Eğrisi', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-0.02, 1.02)
axes[0].set_ylim(-0.02, 1.05)

# --- GRAFİK 2: Precision-Recall Curve ---
for isim, prob, renk in modeller:
    prec, rec, _ = precision_recall_curve(y_te, prob)
    ap = average_precision_score(y_te, prob)
    ls = '-' if isim == 'Ensemble' else '--'
    lw = 3 if isim == 'Ensemble' else 1.5
    axes[1].plot(rec, prec, color=renk, linestyle=ls, linewidth=lw,
                 label=f'{isim} (AP={ap:.3f})')
# Baselines
baseline = y_te.mean()
axes[1].axhline(y=baseline, color='gray', linestyle=':', alpha=0.7,
                label=f'Baseline (pozitif oran={baseline:.2f})')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Grafik 2 — Precision-Recall Eğrisi', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(-0.02, 1.02)
axes[1].set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.savefig('../figures/fig_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 1-2 kaydedildi: figures/fig_roc_pr.png ✓')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

etiketler = ['Normal', 'Papilödem']

for ax, (isim, prob, renk) in zip(axes, [
    ('ET (En İyi Tek Model)', y_prob_et, 'Blues'),
    ('Ensemble (Kalibreli)', y_prob_ens, 'Oranges')
]):
    y_pred = (prob >= 0.5).astype(int)
    cm = confusion_matrix(y_te, y_pred)
    # Normalize (yüzde)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap=renk,
                xticklabels=etiketler, yticklabels=etiketler,
                ax=ax, cbar=False, linewidths=0.5,
                annot_kws={'size': 13, 'weight': 'bold'})
    
    # Ham sayıları da ekle
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.72, f'(n={cm[i,j]})',
                    ha='center', va='center', fontsize=9, color='dimgray')
    
    f1 = f1_score(y_te, y_pred, average='macro')
    acc = accuracy_score(y_te, y_pred)
    ax.set_xlabel('Tahmin', fontsize=11)
    ax.set_ylabel('Gerçek', fontsize=11)
    ax.set_title(f'Grafik 3 — Confusion Matrix\n{isim}\nF1={f1:.3f} | Acc={acc:.3f}',
                 fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/fig_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 3 kaydedildi: figures/fig_confusion_matrix.png ✓')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sol: ET feature importance (en iyi bireysel model)
fi_et = pd.DataFrame({'feature': feats, 'importance': et.feature_importances_})\
          .sort_values('importance', ascending=True)
fi_rf = pd.DataFrame({'feature': feats, 'importance': rf.feature_importances_})\
          .sort_values('importance', ascending=True)

colors_et = ['salmon' if imp > fi_et['importance'].quantile(0.7) else 'steelblue'
              for imp in fi_et['importance']]

axes[0].barh(fi_et['feature'], fi_et['importance'], color=colors_et, alpha=0.85)
axes[0].set_xlabel('Feature Importance (Gini)', fontsize=10)
axes[0].set_title('Grafik 4a — ET Feature Importance\n(Üst %30 vurgulu)', fontsize=10, fontweight='bold')
axes[0].tick_params(labelsize=8)

# Sağ: RF vs ET karşılaştırması
fi_both = pd.DataFrame({'feature': feats,
                         'ET': et.feature_importances_,
                         'RF': rf.feature_importances_}).sort_values('ET', ascending=False)
x = np.arange(len(feats))
w = 0.35
axes[1].bar(x - w/2, fi_both['ET'], w, label='ET', color='salmon', alpha=0.8)
axes[1].bar(x + w/2, fi_both['RF'], w, label='RF', color='steelblue', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(fi_both['feature'], rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('Feature Importance')
axes[1].set_title('Grafik 4b — ET vs RF Feature Importance', fontsize=10, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/fig_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 4 kaydedildi: figures/fig_feature_importance.png ✓')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Kalibrasyon eğrisi
for isim, prob, renk, ls in [
    ('RF', y_prob_rf, 'steelblue', '--'),
    ('ET', y_prob_et, 'salmon', '--'),
    ('GB', y_prob_gb, 'seagreen', '--'),
    ('Ensemble (Kalibreli)', y_prob_ens, 'darkorange', '-')
]:
    frac_pos, mean_pred = calibration_curve(y_te, prob, n_bins=5, strategy='quantile')
    axes[0].plot(mean_pred, frac_pos, marker='o',
                 color=renk, linestyle=ls,
                 linewidth=2.5 if ls=='-' else 1.5,
                 markersize=7, label=isim)

axes[0].plot([0,1],[0,1], 'k--', alpha=0.5, linewidth=1, label='Mükemmel')
axes[0].set_xlabel('Ortalama Tahmin Edilen Olasılık', fontsize=10)
axes[0].set_ylabel('Gerçek Pozitif Oran', fontsize=10)
axes[0].set_title('Grafik 5a — Kalibrasyon Eğrisi\n(Diyagonale yakın = iyi kalibre)', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=9)

# Sağ: Histogram — olasılık dağılımı (Normal vs Papilödem)
prob_normal = y_prob_ens[y_te == 0]
prob_papil  = y_prob_ens[y_te == 1]

axes[1].hist(prob_normal, bins=15, alpha=0.6, color='steelblue',
             label='Normal', edgecolor='white')
axes[1].hist(prob_papil,  bins=15, alpha=0.6, color='salmon',
             label='Papilödem', edgecolor='white')
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=1.5,
                label='Karar eşiği (0.5)')
axes[1].set_xlabel('Tahmin Edilen Papilödem Olasılığı', fontsize=10)
axes[1].set_ylabel('Frekans', fontsize=10)
axes[1].set_title('Grafik 5b — Ensemble Olasılık Dağılımı\n(Sınıflar arası ayırım)', fontsize=10, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/fig_kalibrasyon_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 5 kaydedildi: figures/fig_kalibrasyon_dagilim.png ✓')


In [ ]:
# 20 split sonuçlarını yükle (06 notebook'tan)
import os
sonuc_path = '../results/ensemble_test_sonuclari.csv'

if os.path.exists(sonuc_path):
    sonuc_df = pd.read_csv(sonuc_path)
    print(f'Önceki sonuçlar yüklendi: {len(sonuc_df)} satır')
else:
    print('Sonuç dosyası bulunamadı — grafik mevcut split üzerinden çizilecek.')
    sonuc_df = None

# Bireysel model F1 skorları (07 notebook'tan)
bireysel_path = '../results/bireysel_model_f1_skorlari.csv'
if os.path.exists(bireysel_path):
    f1_df = pd.read_csv(bireysel_path)
    print(f'Bireysel F1 skorları yüklendi: {f1_df.shape}')
else:
    print('Bireysel model sonuçları bulunamadı — lütfen önce 07_istatistiksel_analiz.ipynb çalıştırın.')
    f1_df = None

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sol: Box plot — RF vs ET vs GB vs Ensemble (20 split)
if f1_df is not None and sonuc_df is not None:
    ensemble_f1 = sonuc_df[sonuc_df.seviye=='sample']['macro_f1'].values
    compare_df = f1_df[['RF','ET','GB']].copy()
    compare_df['Ensemble'] = ensemble_f1[:len(compare_df)] if len(ensemble_f1) >= len(compare_df) else np.nan
    
    long = compare_df.melt(var_name='Model', value_name='Macro F1')
    palette = {'RF':'steelblue','ET':'salmon','GB':'seagreen','Ensemble':'darkorange'}
    sns.boxplot(data=long, x='Model', y='Macro F1', palette=palette, ax=axes[0], width=0.5)
    sns.stripplot(data=long, x='Model', y='Macro F1',
                  color='black', alpha=0.3, size=4, jitter=True, ax=axes[0])
    
    for i, (m, col) in enumerate([('RF','steelblue'),('ET','salmon'),('GB','seagreen'),('Ensemble','darkorange')]):
        vals = compare_df[m].dropna()
        axes[0].text(i, vals.median() + 0.01, f'{vals.median():.3f}',
                     ha='center', fontsize=9, fontweight='bold', color=col)
    
    axes[0].set_title('Grafik 6a — Model Karşılaştırma (20 Split)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Macro F1')
    axes[0].set_ylim(0, 1.1)
else:
    axes[0].text(0.5, 0.5, 'Önce 06 ve 07 notebookları çalıştırın',
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=11)
    axes[0].set_title('Grafik 6a — Model Karşılaştırma (20 Split)', fontweight='bold')

# Sağ: Radar/Bar — ortalama metrikler karşılaştırması
metrikler_goster = ['accuracy', 'macro_f1', 'roc_auc', 'pr_auc']
metrik_isimler   = ['Accuracy', 'Macro F1', 'ROC-AUC', 'PR-AUC']

if sonuc_df is not None and f1_df is not None:
    ens_sample = sonuc_df[sonuc_df.seviye=='sample'][metrikler_goster].mean()
    x = np.arange(len(metrikler_goster))
    w = 0.2
    for offset, (model_adi, col) in enumerate([('RF','steelblue'),('ET','salmon'),('GB','seagreen')]):
        # RF/ET/GB için sadece F1 var, diğerleri yaklaşık
        bars = [f1_df[model_adi].mean() if m == 'macro_f1' else ens_sample[m]
                for m in metrikler_goster]
        axes[1].bar(x + offset*w, bars, w, label=model_adi, color=col, alpha=0.8)
    axes[1].bar(x + 3*w, [ens_sample[m] for m in metrikler_goster], w,
                label='Ensemble', color='darkorange', alpha=0.8)
    axes[1].set_xticks(x + 1.5*w)
    axes[1].set_xticklabels(metrik_isimler)
    axes[1].set_ylabel('Skor')
    axes[1].set_ylim(0, 1.15)
    axes[1].set_title('Grafik 6b — Ortalama Metrik Karşılaştırması', fontsize=11, fontweight='bold')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Sonuç dosyaları gerekli',
                 ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Grafik 6b')

plt.tight_layout()
plt.savefig('../figures/fig_model_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 6 kaydedildi: figures/fig_model_karsilastirma.png ✓')


In [ ]:
print('=' * 60)
print('PERFORMANS GRAFİKLERİ — ÖZET')
print('=' * 60)
print('Üretilen grafikler:')
print('  1. figures/fig_roc_pr.png          — ROC + PR eğrisi')
print('  2. figures/fig_confusion_matrix.png — Confusion matrix (ET + Ensemble)')
print('  3. figures/fig_feature_importance.png — Feature importance')
print('  4. figures/fig_kalibrasyon_dagilim.png — Kalibrasyon + olasılık dağılımı')
print('  5. figures/fig_model_karsilastirma.png — Model karşılaştırma')
print('='*60)
print('\nSıradaki adım → Final raporu (PDF)')
